# TP2 - Construire l'auto-attention

On passe d'une moyenne causale a une attention dont les poids sont appris par Q, K et V.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)


## 1. Agregation et moyenne causale

Une ligne de poids indique la contribution des positions visibles et sa somme vaut 1.

In [2]:
a = torch.tril(torch.ones(3, 3))
a = a / a.sum(1, keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()
print(a @ b)

xbow_loop = torch.zeros_like(x)
for b in range(B):
    for t in range(T):
        xbow_loop[b, t] = x[b, :t + 1].mean(dim=0)

wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow_matrix = wei @ x
assert torch.allclose(xbow_loop, xbow_matrix, atol=1e-6)
print('moyenne matricielle:', xbow_matrix.shape)


tensor([[5.0000, 5.0000],
        [6.0000, 4.5000],
        [5.3333, 6.0000]])
moyenne matricielle: torch.Size([4, 8, 32])


## 2. Masque causal et Softmax

Les `-inf` deviennent des probabilites nulles apres Softmax.

In [3]:
tril = torch.tril(torch.ones(T, T))
masked = torch.zeros(T, T).masked_fill(tril == 0, float('-inf'))
wei_masked = F.softmax(masked, dim=-1)
xbow_softmax = wei_masked @ x
assert torch.allclose(xbow_loop, xbow_softmax, atol=1e-6)
assert torch.allclose(wei_masked.sum(-1), torch.ones(T))
print('poids causaux valides:', wei_masked[0])


poids causaux valides: tensor([1., 0., 0., 0., 0., 0., 0., 0.])


## 3. Une tete d'auto-attention apprise

`Q` exprime la demande d'une position, `K` sa compatibilite et `V` l'information effectivement agregée. La division par `sqrt(dh)` stabilise la variance.

In [4]:
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k, q, v = key(x), query(x), value(x)
scores = q @ k.transpose(-2, -1) * head_size ** -0.5
scores = scores.masked_fill(tril == 0, float('-inf'))
attention = F.softmax(scores, dim=-1)
out = attention @ v
print('q,k,v:', q.shape, k.shape, v.shape)
print('wei:', attention.shape, 'out:', out.shape)
print('somme:', attention[0].sum(-1))
assert torch.allclose(attention.triu(1), torch.zeros_like(attention.triu(1)))
assert torch.allclose(attention.sum(-1), torch.ones(B, T), atol=1e-6)


q,k,v: torch.Size([4, 8, 16]) torch.Size([4, 8, 16]) torch.Size([4, 8, 16])
wei: torch.Size([4, 8, 8]) out: torch.Size([4, 8, 16])
somme: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)


In [5]:
raw = (torch.randn(B, T, head_size) @ torch.randn(B, head_size, T))
scaled = raw * head_size ** -0.5
print('variance brute/scaled:', raw.var().item(), scaled.var().item())
z = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print('softmax:', F.softmax(z, -1))
print('softmax sharp:', F.softmax(z * 8, -1))


variance brute/scaled: 16.02717399597168 1.00169837474823
softmax: tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])
softmax sharp: tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])


### Questions et reponses

1. **Comment calculer `c` ?** Chaque ligne de `a` forme une combinaison convexe des lignes de `b`; par exemple la deuxieme sortie est `(b[0] + b[1]) / 2`. Les lignes somment a 1 pour conserver une echelle comparable aux vecteurs agreges.
2. **Quelles sont les formes de `xprev` ?** Pour `t=0`, `(1,C)`; pour `t=3`, `(4,C)`; pour `t=7`, `(8,C)`. La moyenne porte sur `dim=0`, la dimension temporelle de cette tranche, et non sur les features.
3. **Pourquoi `wei @ x` fonctionne-t-il ?** `(T,T) @ (B,T,C)` applique la meme matrice de poids a chaque batch: la dimension `T` des colonnes de `wei` s'aligne avec la dimension temporelle de `x`, donnant `(B,T,C)`.
4. **Que fait `-inf` dans Softmax ?** `exp(-inf)=0`, donc les positions futures ont exactement un poids nul. La somme de chaque ligne reste 1.
5. **Quelle ligne correspond au dernier token ?** `attention[0, -1]`: elle contient les contributions du dernier token et de toutes les positions precedentes, jamais du futur. Sa forme est `(T,)` et sa somme vaut 1.
6. **Pourquoi melanger `V` et non `K` ?** Les keys servent a calculer la compatibilite avec la query; les values contiennent les representations a transmettre. Separer adressage (`Q,K`) et contenu (`V`) permet d'apprendre les deux roles.
7. **Pourquoi diviser par `sqrt(dh)` ?** La variance de `q @ k.T` croît approximativement comme `dh`. La mise a l'echelle la ramene vers 1, evite un Softmax sature et conserve des gradients informatifs.
8. **Que representent les dimensions ?** `B` est le batch, `T` la longueur du contexte, `C` la dimension d'entree et `dh` la dimension d'une tete. Ainsi `q,k,v` sont `(B,T,dh)`, `wei` est `(B,T,T)` et la sortie est `(B,T,dh)`.